# Experiment 1 (v3) — Dataset and analytical baselines (LET, KF-LET)
**Paper:** Learning-based residual link lifetime prediction for UAV swarms (*Drones*, MDPI)

**v3 fixes (code audit):** causal link state (no future information in inputs); every label has a full
60-s follow-up (episodes simulated for 660 s, samples only in the first 600 s); full 12-point KF grid;
6 test episodes per configuration.

What this notebook does (CPU is enough, ~40–50 min on Colab):
1. Writes the simulator and baselines and runs the unit tests (`self_test`).
2. Generates the dataset: **3 mobility models × 2 link models × 23 episodes**
   (14 train / 3 validation / 6 test).
3. Tunes KF-LET on the validation episodes (12 settings) and evaluates LET and KF-LET on the test episodes.
4. Saves the results and downloads **`exp1_results.zip`** — send it to me.

In [ ]:
%%writefile uav_swarm_sim.py
"""
UAV Swarm Link-Lifetime Dataset Generator
=========================================
Paper: Learning-Based Residual Link Lifetime Prediction for UAV Swarms
Author: Renat Abdirayimov (TATU)
Version: 2 (causal link state, fully observed lifetime labels)

What this module does
---------------------
1. Moves N UAVs in 3D using three mobility models:
   - Gauss-Markov (GM): each UAV flies independently, smooth random motion
   - RPGM (Reference Point Group Mobility): UAVs fly in groups (swarm squads)
   - RWP (3D random waypoint): straight flights between random waypoints
2. Computes the air-to-air (A2A) channel for every UAV pair:
   path loss + time-correlated shadowing + Rician small-scale fading -> SNR
3. Decides which links are UP (large-scale SNR >= threshold) -> a dynamic graph.
   The link state is CAUSAL: it depends only on the current and past slots.
4. Labels every UP link with its TRUE residual lifetime (seconds until it breaks).
   The episode is simulated for T + H seconds, but samples are taken only in
   [0, T), so every label has a full H-second follow-up.
5. Computes the classic analytical baseline: Link Expiration Time (LET)
"""
import numpy as np

# ----------------------------------------------------------------------------
# 1. Simulation parameters (these go into Table I of the paper)
# ----------------------------------------------------------------------------
DEFAULT_CFG = dict(
    n_uav=30,              # number of UAVs
    area=(1500.0, 1500.0), # x, y size [m]
    alt=(50.0, 150.0),     # min/max altitude [m]
    dt=0.5,                # time step [s]
    T=600.0,               # episode length [s]
    mobility="rpgm",       # "gm", "rpgm" or "rwp"
    group_size=5,          # UAVs per group (RPGM)
    v_mean=12.0,           # mean speed [m/s]
    v_max=35.0,            # max UAV speed [m/s]
    gm_alpha=0.85,         # Gauss-Markov memory (0=random walk, 1=straight line)
    group_radius=250.0,    # max deviation of a member from its group centre [m]
    member_alpha=0.95,     # memory of a member's own motion inside the group
    member_noise=2.0,      # random acceleration of members [m/s per step]
    member_pull=0.002,     # spring force pulling members back to the centre
    # ---- channel (A2A link) ----
    fc=2.4e9,              # carrier frequency [Hz]
    pt_dbm=10.0,           # transmit power [dBm]
    bw=20e6,               # bandwidth [Hz]
    nf_db=7.0,             # receiver noise figure [dB]
    pl_exp=2.2,            # A2A path-loss exponent
    shadow_std=2.0,        # shadowing std [dB]
    shadow_dcorr=50.0,     # shadowing decorrelation distance [m]
    shadow_model="link",   # "link": independent per link (Gudmundson in distance change)
                           # "spatial": static 2-D shadowing field shared by all links
    field_corr=100.0,      # correlation length of the spatial shadowing field [m]
    rician_k_db=7.6,       # Rician K-factor, measured A2A value (Ad Hoc Networks 2025)
    snr_th_db=10.0,        # link is UP if large-scale SNR >= this [dB]
    break_steps=2,         # link is broken only if down for >= 2 steps (1 s)
    horizon=60.0,          # residual lifetime is capped (censored) here [s]
    channel="a2a",         # "a2a" (SNR threshold, shadowing + Rician) or "disk"
    rwp_pause_max=2.0,     # RWP pause time at a waypoint, U(0, max) [s]
    obs_pos_std=2.0,       # GPS position error in Hello messages [m]
    obs_vel_std=0.3,       # velocity error in Hello messages [m/s]
    seed=0,
)

C = 3e8


def noise_dbm(cfg):
    return -174.0 + 10 * np.log10(cfg["bw"]) + cfg["nf_db"]


def path_loss_db(d, cfg):
    """Log-distance path loss with free-space reference at 1 m."""
    pl0 = 20 * np.log10(4 * np.pi * cfg["fc"] / C)  # FSPL at 1 m
    return pl0 + 10 * cfg["pl_exp"] * np.log10(np.maximum(d, 1.0))


def comm_range(cfg):
    """Distance where mean SNR (no shadowing) equals the threshold."""
    pl_max = cfg["pt_dbm"] - noise_dbm(cfg) - cfg["snr_th_db"]
    pl0 = 20 * np.log10(4 * np.pi * cfg["fc"] / C)
    return 10 ** ((pl_max - pl0) / (10 * cfg["pl_exp"]))


# ----------------------------------------------------------------------------
# 2. Mobility models
# ----------------------------------------------------------------------------
def _reflect(pos, vel, mean_dir, lo, hi):
    """Bounce UAVs off the borders of the area (velocity and mean direction flip)."""
    for k in range(3):
        below, above = pos[:, k] < lo[k], pos[:, k] > hi[k]
        pos[below, k] = 2 * lo[k] - pos[below, k]
        pos[above, k] = 2 * hi[k] - pos[above, k]
        vel[below | above, k] *= -1
        mean_dir[below | above, k] *= -1
    return pos, vel, mean_dir


def _gauss_markov_step(vel, mean_dir, cfg, rng):
    a = cfg["gm_alpha"]
    noise = rng.normal(0, 1, vel.shape) * cfg["v_mean"] * 0.5
    noise[:, 2] *= 0.2  # vertical motion is smaller
    return a * vel + (1 - a) * mean_dir + np.sqrt(1 - a ** 2) * noise


def simulate_mobility(cfg, rng):
    """Returns positions and velocities, shape (n_steps, n_uav, 3)."""
    n, dt = cfg["n_uav"], cfg["dt"]
    steps = int(cfg["T"] / dt)
    lo = np.array([0, 0, cfg["alt"][0]])
    hi = np.array([cfg["area"][0], cfg["area"][1], cfg["alt"][1]])

    def rand_heading(m):
        th = rng.uniform(0, 2 * np.pi, m)
        v = np.stack([np.cos(th), np.sin(th), np.zeros(m)], 1) * cfg["v_mean"]
        return v

    P = np.zeros((steps, n, 3))
    V = np.zeros((steps, n, 3))

    if cfg["mobility"] == "gm":
        pos = rng.uniform(lo, hi, (n, 3))
        vel = rand_heading(n)
        mean_dir = vel.copy()
        for t in range(steps):
            P[t], V[t] = pos, vel
            vel = _gauss_markov_step(vel, mean_dir, cfg, rng)
            pos = pos + vel * dt
            pos, vel, mean_dir = _reflect(pos, vel, mean_dir, lo, hi)
    elif cfg["mobility"] == "rpgm":
        g = cfg["group_size"]
        n_groups = int(np.ceil(n / g))
        member_group = np.arange(n) // g
        margin = np.array([200.0, 200.0, 0.0])
        c_pos = rng.uniform(lo + margin, hi - margin, (n_groups, 3))
        c_pos[:, 2] = rng.uniform(*cfg["alt"], n_groups)
        c_vel = rand_heading(n_groups)
        c_mean = c_vel.copy()
        # each member keeps a random offset that slowly drifts (Gauss-Markov)
        offset = rng.normal(0, cfg["group_radius"] / 2, (n, 3))
        offset[:, 2] *= 0.3
        off_vel = np.zeros((n, 3))
        prev = c_pos[member_group] + offset
        for t in range(steps):
            pos = np.clip(c_pos[member_group] + offset, lo, hi)
            if t > 0:  # physical speed limit (removes jumps at the borders)
                step = pos - prev
                sp = np.linalg.norm(step, axis=1, keepdims=True) / dt
                pos = prev + np.where(sp > cfg["v_max"], step * cfg["v_max"] / np.maximum(sp, 1e-9), step)
                vel = (pos - prev) / dt
            else:
                vel = c_vel[member_group].copy()
            P[t], V[t] = pos, vel
            prev = pos
            # group centres move with Gauss-Markov
            c_vel = _gauss_markov_step(c_vel, c_mean, cfg, rng)
            c_pos = c_pos + c_vel * dt
            c_pos, c_vel, c_mean = _reflect(c_pos, c_vel, c_mean, lo, hi)
            # members wander inside the group (pulled back to the centre)
            off_vel = cfg["member_alpha"] * off_vel + rng.normal(0, cfg["member_noise"], (n, 3)) \
                - cfg["member_pull"] * offset
            off_vel[:, 2] *= 0.5
            offset = offset + off_vel * dt
            r = np.linalg.norm(offset, axis=1, keepdims=True)
            offset = np.where(r > cfg["group_radius"], offset * cfg["group_radius"] / r, offset)
    elif cfg["mobility"] == "rwp":
        # 3-D random waypoint: fly straight to a random waypoint at a random
        # speed, pause, pick the next waypoint
        pos = rng.uniform(lo, hi, (n, 3))
        wp = rng.uniform(lo, hi, (n, 3))
        speed = rng.uniform(0.5, 1.5, n) * cfg["v_mean"]
        pause = np.zeros(n)
        for t in range(steps):
            d = wp - pos
            dist = np.linalg.norm(d, axis=1)
            moving = pause <= 0
            vel = np.where(moving[:, None], d / np.maximum(dist, 1e-9)[:, None] * speed[:, None], 0.0)
            arrive = moving & (dist <= speed * dt)
            vel[arrive] = d[arrive] / dt
            P[t], V[t] = pos, vel
            pos = pos + vel * dt
            pause = pause - dt
            # arrivals: pause, then new waypoint and speed
            pause[arrive] = rng.uniform(0, cfg["rwp_pause_max"], arrive.sum())
            wp[arrive] = rng.uniform(lo, hi, (arrive.sum(), 3))
            speed[arrive] = rng.uniform(0.5, 1.5, arrive.sum()) * cfg["v_mean"]
    else:
        raise ValueError("mobility must be 'gm', 'rpgm' or 'rwp'")
    return P, V


# ----------------------------------------------------------------------------
# 3. A2A channel
# ----------------------------------------------------------------------------
def simulate_channel(P, cfg, rng):
    """Returns distance, large-scale SNR and measured (faded) SNR for all pairs."""
    steps, n, _ = P.shape
    iu, ju = np.triu_indices(n, 1)
    diff = P[:, iu, :] - P[:, ju, :]
    dist = np.linalg.norm(diff, axis=2)                       # (steps, n_pairs)

    if cfg.get("shadow_model", "link") == "spatial":
        shadow = spatial_shadowing(P, iu, ju, cfg, rng)
        return _finish_channel(dist, shadow, iu, ju, cfg, rng)
    # time-correlated shadowing (Gudmundson model): rho = exp(-moved / d_corr)
    shadow = np.zeros_like(dist)
    shadow[0] = rng.normal(0, cfg["shadow_std"], dist.shape[1])
    for t in range(1, steps):
        moved = np.abs(dist[t] - dist[t - 1]) + 1e-3
        rho = np.exp(-moved / cfg["shadow_dcorr"])
        shadow[t] = rho * shadow[t - 1] + np.sqrt(1 - rho ** 2) * rng.normal(
            0, cfg["shadow_std"], dist.shape[1])

    return _finish_channel(dist, shadow, iu, ju, cfg, rng)


def spatial_shadowing(P, iu, ju, cfg, rng, res=10.0, n_pts=8):
    """Spatially correlated shadowing: a static 2-D Gaussian random field G(x, y)
    with correlation length `field_corr` (buildings / terrain). The shadowing of
    link (i, j) is the field averaged along the ground projection of the segment
    i-j, rescaled to std `shadow_std`. Links that cross the same region are
    therefore correlated, and a link's shadowing changes as its endpoints move."""
    from scipy.ndimage import gaussian_filter
    nx = int(cfg["area"][0] / res) + 1
    ny = int(cfg["area"][1] / res) + 1
    G = gaussian_filter(rng.normal(size=(ny, nx)), sigma=cfg["field_corr"] / res / 2, mode="wrap")
    G = (G - G.mean()) / G.std()
    s = np.linspace(0.0, 1.0, n_pts)
    pi, pj = P[:, iu, :2], P[:, ju, :2]                          # (T, M, 2)
    pts = pi[:, :, None, :] + s[None, None, :, None] * (pj - pi)[:, :, None, :]
    ix = np.clip(np.rint(pts[..., 0] / res).astype(int), 0, nx - 1)
    iy = np.clip(np.rint(pts[..., 1] / res).astype(int), 0, ny - 1)
    L = G[iy, ix].mean(-1)                                       # (T, M)
    return cfg["shadow_std"] * L / L.std()


def _finish_channel(dist, shadow, iu, ju, cfg, rng):
    snr_ls = cfg["pt_dbm"] - path_loss_db(dist, cfg) - shadow - noise_dbm(cfg)

    # Rician fading power |h|^2 with unit mean
    K = 10 ** (cfg["rician_k_db"] / 10)
    los = np.sqrt(K / (K + 1))
    nlos = np.sqrt(1 / (2 * (K + 1)))
    h = los + nlos * (rng.normal(size=dist.shape) + 1j * rng.normal(size=dist.shape))
    snr_meas = snr_ls + 10 * np.log10(np.abs(h) ** 2)
    return dist, snr_ls, snr_meas, iu, ju


# ----------------------------------------------------------------------------
# 4. Labels: true residual link lifetime + LET baseline
# ----------------------------------------------------------------------------
def debounce(up_raw, k):
    """CAUSAL link state machine (uses only the current and past slots).
    An active link is declared BROKEN at the k-th consecutive slot whose
    large-scale SNR is below the threshold (k missed Hello messages); during
    the first k-1 low slots it is still active. A broken link becomes active
    again as soon as the threshold is exceeded. Initial state = up_raw[0]."""
    if k <= 1:
        return up_raw.copy()
    out = np.zeros_like(up_raw, dtype=bool)
    state = up_raw[0].copy()
    miss = np.where(up_raw[0], 0, k).astype(int)
    out[0] = state
    for t in range(1, up_raw.shape[0]):
        miss = np.where(up_raw[t], 0, miss + 1)
        state = np.where(up_raw[t], True, state & (miss < k))
        out[t] = state
    return out


def residual_lifetime(up, cfg):
    """For every (t, pair) that is UP, seconds until the link first goes DOWN.
    Capped at `horizon` (right-censored). Returns (lifetime, censored_flag)."""
    steps, m = up.shape
    dt, H = cfg["dt"], cfg["horizon"]
    life = np.zeros((steps, m))
    nxt = np.full(m, np.inf)  # steps until next DOWN, scanning backwards
    for t in range(steps - 1, -1, -1):
        nxt = np.where(up[t], nxt + 1, 0)
        life[t] = nxt
    life = life * dt
    censored = life >= H
    life = np.minimum(life, H)
    life[~up] = 0.0
    return life, censored & up


def link_expiration_time(P, V, iu, ju, r, cfg):
    """Classic LET (Su, Lee & Gerla, 2001) extended to 3D: assumes both UAVs
    keep their current velocity. Solves |dp + dv*tau| = r for tau > 0."""
    dp = P[:, iu, :] - P[:, ju, :]
    dv = V[:, iu, :] - V[:, ju, :]
    a = np.sum(dv * dv, -1)
    b = 2 * np.sum(dp * dv, -1)
    c = np.sum(dp * dp, -1) - r ** 2
    disc = np.maximum(b ** 2 - 4 * a * c, 0)
    with np.errstate(divide="ignore", invalid="ignore"):
        tau = (-b + np.sqrt(disc)) / (2 * a)
    tau = np.where(a < 1e-9, cfg["horizon"], tau)
    tau = np.where(c > 0, 0.0, tau)  # already out of range
    return np.clip(tau, 0, cfg["horizon"])


# ----------------------------------------------------------------------------
# 5. One full episode -> dictionary of arrays
# ----------------------------------------------------------------------------
def generate_episode(cfg=None, **overrides):
    cfg = {**DEFAULT_CFG, **(cfg or {}), **overrides}
    rng = np.random.default_rng(cfg["seed"])
    n_keep = int(round(cfg["T"] / cfg["dt"]))            # sample slots [0, T)
    sim_cfg = {**cfg, "T": cfg["T"] + cfg["horizon"] + cfg["dt"]}   # + follow-up
    P, V = simulate_mobility(sim_cfg, rng)
    dist, snr_ls, snr_meas, iu, ju = simulate_channel(P, cfg, rng)
    if cfg["channel"] == "disk":          # classical distance-threshold link
        up_raw = dist <= comm_range(cfg)
    elif cfg["channel"] == "a2a":         # SNR threshold with shadowing
        up_raw = snr_ls >= cfg["snr_th_db"]
    else:
        raise ValueError("channel must be 'a2a' or 'disk'")
    up = debounce(up_raw, cfg["break_steps"])
    life, cens = residual_lifetime(up, cfg)
    # Hello-message observations: GPS position/velocity errors
    P_obs = P + rng.normal(0, cfg["obs_pos_std"], P.shape)
    V_obs = V + rng.normal(0, cfg["obs_vel_std"], V.shape)
    let = link_expiration_time(P_obs, V_obs, iu, ju, comm_range(cfg), cfg)
    # every kept label has a full horizon of follow-up inside the simulation
    assert up.shape[0] >= n_keep + int(round(cfg["horizon"] / cfg["dt"])) + 1
    cut = lambda x: x[:n_keep]
    P, V, P_obs, V_obs, dist, snr_ls, snr_meas, up, life, cens, let = map(
        cut, (P, V, P_obs, V_obs, dist, snr_ls, snr_meas, up, life, cens, let))
    assert np.isfinite(life).all()
    return dict(cfg=cfg, pos=P, vel=V, pos_obs=P_obs, vel_obs=V_obs, dist=dist,
                snr_ls=snr_ls, snr_meas=snr_meas, up=up, life=life, censored=cens,
                let=let, iu=iu, ju=ju)


def self_test(verbose=True):
    """Unit tests for the two properties checked in the code audit."""
    k = 2
    f = lambda seq: debounce(np.array(seq, bool)[:, None], k)[:, 0].astype(int).tolist()
    assert f([1, 0, 1]) == [1, 1, 1]          # 1-slot dip: still active
    assert f([1, 0, 0]) == [1, 1, 0]          # break declared at the 2nd miss
    assert f([1, 0, 0, 1]) == [1, 1, 0, 1]    # reconnection
    assert f([0, 0, 1]) == [0, 0, 1]          # starts down
    # causality: same history, different futures -> identical states so far
    rng = np.random.default_rng(1)
    for _ in range(200):
        a = rng.random((40, 5)) > 0.3
        b = a.copy(); t = rng.integers(1, 39); b[t + 1:] = rng.random((39 - t, 5)) > 0.3
        assert (debounce(a, k)[:t + 1] == debounce(b, k)[:t + 1]).all()
    # labels: every sample has a full follow-up; censored <=> life == H
    ep = generate_episode(mobility="gm", seed=123, T=60.0)
    H = ep["cfg"]["horizon"]
    assert ep["up"].shape[0] == 120
    assert ((ep["life"] >= H) == ep["censored"])[ep["up"]].all()
    # a label at t must not depend on anything after t + H (finite follow-up)
    cfg = {**DEFAULT_CFG, "dt": 0.5, "horizon": 5.0}
    nH = 10
    for _ in range(50):
        u = rng.random((80, 4)) > 0.1
        full, _ = residual_lifetime(u, cfg)
        for t in rng.integers(0, 80 - nH - 1, 5):
            part, _ = residual_lifetime(u[:t + nH + 1], cfg)
            assert np.allclose(full[t], part[t])
    if verbose:
        print("self_test passed: causal link state, full-horizon labels")


if __name__ == "__main__":
    ep = generate_episode()
    cfg = ep["cfg"]
    up = ep["up"]
    print(f"Comm range R = {comm_range(cfg):.1f} m, noise = {noise_dbm(cfg):.1f} dBm")
    print(f"Steps = {up.shape[0]}, pairs = {up.shape[1]}")
    print(f"Avg active links per snapshot = {up.sum(1).mean():.1f}")
    print(f"Avg node degree = {2*up.sum(1).mean()/cfg['n_uav']:.2f}")
    m = up & ~ep["censored"]
    print(f"Uncensored samples = {m.sum()}, mean lifetime = {ep['life'][m].mean():.1f} s")
    err = np.abs(ep["let"][m] - ep["life"][m])
    print(f"LET baseline MAE = {err.mean():.2f} s")

In [ ]:
%%writefile baselines.py
"""
Analytical baselines and evaluation metrics for residual link lifetime.

- LET    : constant-velocity link expiration time (Su, Lee & Gerla, 2001)
- KF-LET : Kalman filter with a Singer (manoeuvring-target) acceleration model,
           as in Kalman-assisted FANET routing (MP-QGRD, KFQR); the filtered
           trajectories are extrapolated and the first time the distance
           exceeds the range R is the predicted lifetime.
"""
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score


# ---------------------------------------------------------------------------
# Kalman filter (Singer model), identical for every UAV and axis -> vectorised
# ---------------------------------------------------------------------------
def singer_matrices(dt, tau, q):
    a = 1.0 / tau
    e = np.exp(-a * dt)
    F = np.array([[1, dt, (a * dt - 1 + e) / a**2],
                  [0, 1, (1 - e) / a],
                  [0, 0, e]])
    Q = q * np.array([[dt**5 / 20, dt**4 / 8, dt**3 / 6],
                      [dt**4 / 8, dt**3 / 3, dt**2 / 2],
                      [dt**3 / 6, dt**2 / 2, dt]])
    return F, Q


def kalman_singer(P_obs, V_obs, cfg, tau=3.0, q=2.0):
    """Filter noisy positions/velocities. Inputs (T, N, 3).
    Returns filtered position, velocity, acceleration (T, N, 3)."""
    T, N, _ = P_obs.shape
    dt = cfg["dt"]
    F, Q = singer_matrices(dt, tau, q)
    H = np.array([[1.0, 0, 0], [0, 1.0, 0]])
    R = np.diag([cfg["obs_pos_std"]**2, cfg["obs_vel_std"]**2])
    x = np.zeros((N, 3, 3))                     # (uav, axis, state)
    x[:, :, 0], x[:, :, 1] = P_obs[0], V_obs[0]
    Pc = np.diag([cfg["obs_pos_std"]**2, cfg["obs_vel_std"]**2, 4.0])
    out = np.zeros((T, N, 3, 3))
    for t in range(T):
        if t > 0:                               # predict
            x = x @ F.T
            Pc = F @ Pc @ F.T + Q
        S = H @ Pc @ H.T + R                    # update (same gain for all)
        K = Pc @ H.T @ np.linalg.inv(S)
        z = np.stack([P_obs[t], V_obs[t]], -1)  # (N, 3, 2)
        x = x + (z - x @ H.T) @ K.T
        Pc = (np.eye(3) - K @ H) @ Pc
        out[t] = x
    return out[..., 0], out[..., 1], out[..., 2]


def kf_let(P_obs, V_obs, iu, ju, R, cfg, tau=3.0, q=2.0):
    """Predicted lifetime from Singer-model extrapolation of both UAVs."""
    p, v, a = kalman_singer(P_obs, V_obs, cfg, tau, q)
    H, dt = cfg["horizon"], cfg["dt"]
    tk = np.arange(dt, H + dt / 2, dt)                      # prediction grid
    al = 1.0 / tau
    g = (np.exp(-al * tk) - 1 + al * tk) / al**2            # Singer position gain
    dp, dv, da = p[:, iu] - p[:, ju], v[:, iu] - v[:, ju], a[:, iu] - a[:, ju]
    T, M, _ = dp.shape
    life = np.full((T, M), H)
    for t in range(T):
        traj = dp[t][None] + tk[:, None, None] * dv[t][None] + g[:, None, None] * da[t][None]
        out = np.linalg.norm(traj, axis=2) > R                # (K, M)
        first = np.where(out.any(0), out.argmax(0), -1)
        life[t] = np.where(first >= 0, tk[np.maximum(first, 0)], H)
        life[t][np.linalg.norm(dp[t], axis=1) > R] = 0.0
    return np.clip(life, 0, H)


# ---------------------------------------------------------------------------
# Metrics
# ---------------------------------------------------------------------------
def evaluate(pred, life, censored, up, delta=5.0):
    """Regression metrics on uncensored active links + break-within-delta
    classification on all active links (censored links never break early)."""
    m = up & ~censored
    y, yh = life[m], pred[m]
    err = yh - y
    mae, rmse = np.abs(err).mean(), np.sqrt((err**2).mean())
    r2 = 1 - (err**2).sum() / ((y - y.mean())**2).sum()
    lab = (up & ~censored & (life <= delta))[up]
    score = -pred[up]
    return dict(MAE=mae, RMSE=rmse, R2=r2,
                AUC=roc_auc_score(lab, score), AUPRC=average_precision_score(lab, score),
                n=int(m.sum()), pos_rate=lab.mean())

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, os, json, time, itertools
import importlib, uav_swarm_sim, baselines
importlib.reload(uav_swarm_sim); importlib.reload(baselines)
from uav_swarm_sim import *
from baselines import *
self_test()             # must print "self_test passed"

SAVE_TO_DRIVE = False   # True -> dataset is stored in Google Drive (needed for Experiment 2)
N_EP = 23               # episodes per (mobility, channel)
SPLIT = dict(train=range(0, 14), val=range(14, 17), test=range(17, 23))
MOBILITY = ["rpgm", "gm", "rwp"]
CHANNEL = ["a2a", "disk"]

root = "dataset_v3"
if SAVE_TO_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")
    root = "/content/drive/MyDrive/uav_gnn_paper/dataset_v3"
os.makedirs(root, exist_ok=True); os.makedirs("figures", exist_ok=True)
print("R =", round(comm_range(DEFAULT_CFG), 1), "m")

## 1. Generate the dataset

In [ ]:
t0 = time.time(); stats = []
for mob, ch in itertools.product(MOBILITY, CHANNEL):
    for s in range(N_EP):
        f = f"{root}/{mob}_{ch}_ep{s:02d}.npz"
        if os.path.exists(f):
            continue
        ep = generate_episode(mobility=mob, channel=ch, seed=10_000 + s)
        np.savez_compressed(f,
            pos_obs=ep["pos_obs"].astype(np.float32), vel_obs=ep["vel_obs"].astype(np.float32),
            snr_meas=ep["snr_meas"].astype(np.float16), up=ep["up"],
            life=ep["life"].astype(np.float16), censored=ep["censored"],
            let=ep["let"].astype(np.float16), iu=ep["iu"], ju=ep["ju"], cfg=json.dumps(ep["cfg"]))
        up, m = ep["up"], ep["up"] & ~ep["censored"]
        stats.append(dict(mobility=mob, channel=ch, episode=s, links=up.sum(1).mean(),
                          samples=int(up.sum()), censored=ep["censored"][up].mean(),
                          median_life=np.median(ep["life"][m])))
    print(f"{mob:5s} {ch:5s} done  ({time.time()-t0:.0f} s)")
if stats:
    pd.DataFrame(stats).to_csv("dataset_stats.csv", index=False)
ds = pd.read_csv("dataset_stats.csv")
display(ds.groupby(["mobility", "channel"]).agg(
    avg_links=("links", "mean"), samples=("samples", "sum"),
    censored=("censored", "mean"), median_life_s=("median_life", "mean")).round(2))

## 2. Tune KF-LET on the validation episodes (fair baseline)

In [ ]:
def load(mob, ch, s):
    d = np.load(f"{root}/{mob}_{ch}_ep{s:02d}.npz")
    cfg = json.loads(str(d["cfg"]))
    return d, cfg

grid = list(itertools.product([2, 3, 5, 10], [0.5, 1.0, 2.0]))   # 12 settings (tau [s], q)
tune = []
for mob in MOBILITY:
    for tau, q in grid:
        maes = []
        for s in SPLIT["val"]:
            d, cfg = load(mob, "a2a", s)
            pred = kf_let(d["pos_obs"], d["vel_obs"], d["iu"], d["ju"], comm_range(cfg), cfg, tau, q)
            maes.append(evaluate(pred, d["life"].astype(float), d["censored"], d["up"])["MAE"])
        tune.append(dict(mobility=mob, tau=tau, q=q, val_MAE=np.mean(maes)))
    print(mob, "tuned")
tune = pd.DataFrame(tune)
best = tune.loc[tune.groupby("mobility").val_MAE.idxmin()].set_index("mobility")
display(tune.pivot(index=["tau", "q"], columns="mobility", values="val_MAE").round(2))
print("Best (tau, q) per mobility:"); display(best)

## 3. Evaluate LET and KF-LET on the test episodes

In [ ]:
rows = []
for mob, ch in itertools.product(MOBILITY, CHANNEL):
    tau, q = best.loc[mob, "tau"], best.loc[mob, "q"]
    for s in SPLIT["test"]:
        d, cfg = load(mob, ch, s)
        life, cens, up = d["life"].astype(float), d["censored"], d["up"]
        let = d["let"].astype(float)
        kf = kf_let(d["pos_obs"], d["vel_obs"], d["iu"], d["ju"], comm_range(cfg), cfg, tau, q)
        for name, pred in [("LET", let), ("KF-LET", kf)]:
            r = evaluate(pred, life, cens, up); r.update(method=name, mobility=mob, channel=ch, episode=s)
            rows.append(r)
    print(mob, ch, "done")
res = pd.DataFrame(rows)
res.to_csv("results_exp1.csv", index=False)
summary = (res.groupby(["mobility", "channel", "method"])[["MAE", "RMSE", "R2", "AUC", "AUPRC"]]
              .agg(["mean", "std"]).round(3))
display(summary)

## 4. Figure: effect of the channel model on the analytical predictors

In [ ]:
COLORS = {"LET": "#2a78d6", "KF-LET": "#eb6834"}
fig, axs = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True)
for ax, mob in zip(axs, MOBILITY):
    x = np.arange(len(CHANNEL)); w = 0.36
    for k, m in enumerate(["LET", "KF-LET"]):
        g = res[(res.mobility == mob) & (res.method == m)].groupby("channel").MAE
        mu, sd = g.mean()[CHANNEL], g.std()[CHANNEL]
        ax.bar(x + (k - 0.5) * w, mu, w * 0.92, yerr=sd, color=COLORS[m], label=m, capsize=3)
    ax.set_xticks(x, ["A2A channel", "disk model"]); ax.set_title(mob.upper())
    ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", alpha=0.25)
axs[0].set_ylabel("MAE of residual lifetime [s]"); axs[0].legend(frameon=False)
plt.tight_layout(); fig.savefig("figures/fig_exp1_channel_effect.pdf"); fig.savefig("figures/fig_exp1_channel_effect.png", dpi=300)
plt.show()

## 5. LaTeX rows for the paper and download

In [ ]:
names = {"rpgm": "RPGM", "gm": "Gauss--Markov", "rwp": "RWP"}
with open("table_exp1.tex", "w") as f:
    for mob, ch, m in itertools.product(MOBILITY, CHANNEL, ["LET", "KF-LET"]):
        g = res[(res.mobility == mob) & (res.channel == ch) & (res.method == m)]
        cell = lambda c, p=2: f"{g[c].mean():.{p}f} $\\pm$ {g[c].std():.{p}f}"
        f.write(f"{names[mob]} & {ch.upper() if ch=='a2a' else 'Disk'} & {m} & {cell('MAE')} & {cell('RMSE')} & "
                f"{cell('R2',3)} & {cell('AUC',3)} & {cell('AUPRC',3)} & {g.pos_rate.mean():.3f} \\\\\n")
print(open("table_exp1.tex").read())

import shutil
os.makedirs("exp1_out", exist_ok=True)
for f in ["results_exp1.csv", "dataset_stats.csv", "table_exp1.tex",
          "figures/fig_exp1_channel_effect.pdf", "figures/fig_exp1_channel_effect.png"]:
    shutil.copy(f, "exp1_out")
tune.to_csv("exp1_out/kf_tuning.csv", index=False)
shutil.make_archive("exp1_results", "zip", "exp1_out")
try:
    from google.colab import files; files.download("exp1_results.zip")
except ImportError:
    print("exp1_results.zip created")